# HTGAA CFPS Coopetition — database snapshot and exploratory visualizations

**Repository context:** [RonanChance/htgaa-automation](https://github.com/RonanChance/htgaa-automation) — the existing SvelteKit/PocketBase application and source of truth for the CFPS designer.

This notebook is intended to be committed later at `notebooks/cfps_database_snapshot_and_exploration.ipynb` in that repository. It does **not** push commits automatically.

## Source-aligned details verified in the repository

- PocketBase base URL: `https://opentrons-art-pb.rcdonovan.com`.
- Existing server code authenticates through PocketBase's `_superusers` collection using `PB_EMAIL` and `PB_PASSWORD`. The previous draft's `users` default was not aligned with the repository and is corrected here.
- Community formulation submissions are stored in `cfps_designs`; each record's `design_json` contains `sample_id`, `sample_type`, `reagent_list`, and `metadata` (including the target protein and batch fields where present).
- The reagent catalogue is stored in `cfps_reagent_groups`, record `t7a97pyfs8lx67w`.
- The repo's CFPS contract refers to reagent volumes in nL, a 25 nL grid, and a 20 µL reaction total. The notebook reports volume totals as a diagnostic; it does not discard records that differ from the current contract.

Relevant source files: [`src/lib/server/pb.js`](https://github.com/RonanChance/htgaa-automation/blob/main/src/lib/server/pb.js), [`src/lib/server/loadCommunityDesigns.js`](https://github.com/RonanChance/htgaa-automation/blob/main/src/lib/server/loadCommunityDesigns.js), [`src/lib/server/loadCfpsReagentGroups.js`](https://github.com/RonanChance/htgaa-automation/blob/main/src/lib/server/loadCfpsReagentGroups.js), and [`src/routes/cfps/submit-design/+server.js`](https://github.com/RonanChance/htgaa-automation/blob/main/src/routes/cfps/submit-design/+server.js).

## Security / data handling

This notebook performs **read-only GET requests**, apart from the authentication POST required to obtain a token. It does not submit, update, or delete CFPS designs. Credentials are loaded from Colab Secrets or hidden prompts and are never written to files or printed. The Keycloak client secret is not needed for this direct PocketBase flow and is intentionally not used.

Raw records (which may contain authorship, comments, conversation links, or other member-associated data) are saved only under `/content/cfps_private_snapshots/`, **outside the cloned Git repository**. The `reports/cfps_exploration/` directory inside the repo receives aggregate tables and figures only. Review sharing/redistribution permission with the coordinator before committing even derived outputs. Do not commit raw snapshots or row-level exports.

**Access condition:** the `PB_EMAIL` / `PB_PASSWORD` pair must be credentials authorized to authenticate against `_superusers` on this PocketBase instance, as the repository's server code does. If Ronan supplied ordinary Keycloak login credentials rather than PB superuser credentials, do not try to work around access controls; ask him for the intended approved access route.

In [ ]:
# 1. Clone the canonical application repository into this Colab runtime.
from pathlib import Path
import os
import subprocess
import sys

REPO_URL = "https://github.com/RonanChance/htgaa-automation.git"
REPO_ROOT = Path("/content/htgaa-automation")

if not (REPO_ROOT / ".git").exists():
    if REPO_ROOT.exists() and any(REPO_ROOT.iterdir()):
        raise RuntimeError(
            f"{REPO_ROOT} exists but is not a Git checkout. Rename/remove it, then rerun this cell."
        )
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(REPO_ROOT)], check=True)

REPO_SHA = subprocess.check_output(
    ["git", "-C", str(REPO_ROOT), "rev-parse", "HEAD"], text=True
).strip()

# Only aggregate/publication-oriented outputs are written inside the repository.
REPORT_DIR = REPO_ROOT / "reports" / "cfps_exploration"
REPORT_DIR.mkdir(parents=True, exist_ok=True)

# Private raw data remain outside Git and outside the repository tree.
PRIVATE_ROOT = Path("/content/cfps_private_snapshots")
PRIVATE_ROOT.mkdir(parents=True, exist_ok=True)

# These are standard Colab-friendly Python dependencies; no Node/Svelte build is required.
subprocess.run([sys.executable, "-m", "pip", "-q", "install", "requests", "pandas", "matplotlib"], check=True)

print(f"Repository: {REPO_URL}")
print(f"Checked-out commit: {REPO_SHA[:12]}")
print(f"Aggregate reports will go to: {REPORT_DIR}")
print(f"Private snapshots will go to: {PRIVATE_ROOT} (outside the repo)")


In [ ]:
# 2. Load credentials from Colab Secrets, otherwise prompt without echoing the password.
# Recommended Colab secret names: PB_EMAIL and PB_PASSWORD. Optional: PB_URL.
import os
from getpass import getpass

try:
    from google.colab import userdata
except Exception:
    userdata = None

def secret_or_prompt(key, prompt, hidden=True):
    value = None
    if userdata is not None:
        try:
            value = userdata.get(key)
        except Exception:
            value = None
    if not value:
        value = getpass(prompt) if hidden else input(prompt)
    if not value:
        raise ValueError(f"No value provided for {key}.")
    os.environ[key] = value

# Prefer an authorized PocketBase authentication token when available.
# The platform's Keycloak agent token is NOT automatically a PocketBase JWT.
PB_TOKEN = os.environ.get("PB_TOKEN", "")
if userdata is not None and not PB_TOKEN:
    try:
        PB_TOKEN = userdata.get("PB_TOKEN") or ""
    except Exception:
        PB_TOKEN = ""
if PB_TOKEN:
    print("Using a supplied PocketBase authentication token; value hidden.")
else:
    secret_or_prompt("PB_EMAIL", "PocketBase authorized email / identity: ", hidden=False)
    secret_or_prompt("PB_PASSWORD", "PocketBase authorized password: ", hidden=True)

# Optional PB_URL may be set as a Colab Secret; otherwise use the URL in Ronan's source.
if userdata is not None:
    try:
        pb_url_secret = userdata.get("PB_URL")
    except Exception:
        pb_url_secret = None
    if pb_url_secret:
        os.environ["PB_URL"] = pb_url_secret

PB_URL = os.environ.get("PB_URL", "https://db.rcdonovan.com").rstrip("/")
PB_AUTH_COLLECTION = "_superusers"  # confirmed by src/lib/server/pb.js and tools/sync-cfps.mjs
print(f"Credentials loaded into runtime. API base: {PB_URL}; auth collection: {PB_AUTH_COLLECTION}.")
print("Credential values and authentication tokens will not be displayed.")


In [ ]:
# 3. Read-only PocketBase client: authenticate and fetch paginated collection records.
# No create/update/delete endpoints are called by this class.
import time
import requests
from urllib.parse import quote

class PocketBaseReadOnlyClient:
    def __init__(self, base_url, identity=None, password=None, auth_collection="_superusers", timeout=45, token=None):
        self.base_url = base_url.rstrip("/")
        self.identity = identity
        self.password = password
        self.auth_collection = auth_collection
        self.timeout = timeout
        self.session = requests.Session()
        self.token = token
        if token:
            self.session.headers.update({"Authorization": f"Bearer {token}"})

    def authenticate(self):
        if self.token and not self.identity:
            return {}  # externally supplied PocketBase bearer token
        url = f"{self.base_url}/api/collections/{quote(self.auth_collection, safe='')}/auth-with-password"
        response = self.session.post(
            url,
            json={"identity": self.identity, "password": self.password},
            timeout=self.timeout,
        )
        if not response.ok:
            # Avoid echoing credentials/token; include only status and a shortened server message.
            try:
                detail = response.json().get("message", "")
            except Exception:
                detail = ""
            raise RuntimeError(
                f"PocketBase authentication failed (HTTP {response.status_code}). "
                f"Server message: {str(detail)[:300] or 'no message returned'}. "
                "Confirm these are credentials authorized for _superusers, not merely Keycloak web-login credentials."
            )
        payload = response.json()
        if not payload.get("token"):
            raise RuntimeError("PocketBase authentication response did not contain a token.")
        self.token = payload["token"]
        self.session.headers.update({"Authorization": f"Bearer {self.token}"})
        return payload.get("record", {})

    def _get(self, path, params=None, retry_auth=True):
        if not self.token:
            self.authenticate()
        response = self.session.get(f"{self.base_url}{path}", params=params, timeout=self.timeout)
        if response.status_code == 401 and retry_auth:
            self.authenticate()
            response = self.session.get(f"{self.base_url}{path}", params=params, timeout=self.timeout)
        if not response.ok:
            try:
                body = response.json()
                message = body.get("message", "")
            except Exception:
                message = ""
            raise RuntimeError(
                f"PocketBase GET failed (HTTP {response.status_code}) at {path}. "
                f"Server message: {str(message)[:300] or 'no message returned'}."
            )
        return response.json()

    def get_record(self, collection, record_id):
        return self._get(
            f"/api/collections/{quote(collection, safe='')}/records/{quote(record_id, safe='')}"
        )

    def list_records(self, collection, per_page=200, sort="-created", max_records=None):
        per_page = max(1, min(int(per_page), 500))
        page = 1
        rows = []
        advertised_total = None
        while True:
            params = {"page": page, "perPage": per_page}
            if sort:
                params["sort"] = sort
            payload = self._get(
                f"/api/collections/{quote(collection, safe='')}/records", params=params
            )
            items = payload.get("items", [])
            if advertised_total is None:
                advertised_total = payload.get("totalItems")
            if not isinstance(items, list):
                raise RuntimeError(f"Unexpected PocketBase list response for {collection!r}: items is not a list.")
            rows.extend(items)
            total_pages = int(payload.get("totalPages") or 0)
            if max_records is not None and len(rows) >= max_records:
                rows = rows[:max_records]
                break
            if not items or (total_pages and page >= total_pages):
                break
            page += 1
            time.sleep(0.05)  # gentle pagination; does not materially slow small snapshots
        result = {
            "collection": collection,
            "records": rows,
            "advertised_total_items": advertised_total,
            "records_fetched": len(rows),
            "possibly_truncated": bool(max_records is not None and advertised_total is not None and len(rows) < advertised_total),
        }
        return result

client = PocketBaseReadOnlyClient(
    PB_URL, os.environ.get("PB_EMAIL"), os.environ.get("PB_PASSWORD"), PB_AUTH_COLLECTION, token=PB_TOKEN
)
identity_record = client.authenticate()
print("Authenticated successfully; token omitted.")
print("Authenticated record ID:", identity_record.get("id", "not returned"))


## 4. Smoke-test only the source-confirmed collections

The notebook does not enumerate or dump every collection. It begins with the two collections identified in the application source:

- `cfps_designs`: the submitted CFPS formulations. The application stores recipe data in `design_json.reagent_list` and formulation metadata in `design_json.metadata`.
- `cfps_reagent_groups` / `t7a97pyfs8lx67w`: the reagent catalogue record used by the designer.

The `cfps_designs` list is intentionally login-gated in the application. Use this extraction only under the database access and data-sharing permission Ronan gave you. A successful superuser login does not imply that raw records may be republished.

In [ ]:
# 4. Probe one list item and the known reagent-catalogue record before full retrieval.
# All requests below are GET after authentication.
DESIGNS_COLLECTION = "cfps_designs"
REAGENT_GROUPS_COLLECTION = "cfps_reagent_groups"
REAGENT_GROUPS_RECORD_ID = "t7a97pyfs8lx67w"

sample_page = client.list_records(DESIGNS_COLLECTION, per_page=1, max_records=1)
reagent_group_probe = client.get_record(REAGENT_GROUPS_COLLECTION, REAGENT_GROUPS_RECORD_ID)

print("cfps_designs sample returned:", sample_page["records_fetched"])
print("cfps_designs advertised total (if available):", sample_page["advertised_total_items"])
print("Sample record keys:", sorted(sample_page["records"][0].keys()) if sample_page["records"] else "no rows visible")
print("Reagent-group record ID returned:", reagent_group_probe.get("id", "missing id"))
print("Reagent-group record keys:", sorted(reagent_group_probe.keys()))


In [ ]:
# 5. Fetch a read-only snapshot and store raw responses outside the cloned repository.
# Change MAX_DESIGNS only for a deliberate pilot; set it to None for all accessible records.
from datetime import datetime, timezone
import json

MAX_DESIGNS = None  # e.g. 25 for a small test; None fetches all pages returned to this account
SNAPSHOT_TIME = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
PRIVATE_RUN_DIR = PRIVATE_ROOT / SNAPSHOT_TIME
PRIVATE_RUN_DIR.mkdir(parents=True, exist_ok=False)

designs_result = client.list_records(
    DESIGNS_COLLECTION, per_page=200, sort="-created", max_records=MAX_DESIGNS
)
reagent_groups_record = client.get_record(REAGENT_GROUPS_COLLECTION, REAGENT_GROUPS_RECORD_ID)

snapshot = {
    "metadata": {
        "created_utc": datetime.now(timezone.utc).isoformat(),
        "source_repository": REPO_URL,
        "source_repository_commit": REPO_SHA,
        "pocketbase_base_url": PB_URL,
        "authentication_collection": PB_AUTH_COLLECTION,
        "read_only_snapshot": True,
        "collections_requested": [DESIGNS_COLLECTION, REAGENT_GROUPS_COLLECTION],
        "designs_advertised_total_items": designs_result["advertised_total_items"],
        "designs_records_fetched": designs_result["records_fetched"],
        "designs_possibly_truncated": designs_result["possibly_truncated"],
        "note": "Raw records are private working data; do not commit or publish without coordinator approval."
    },
    "collections": {
        DESIGNS_COLLECTION: designs_result["records"],
        REAGENT_GROUPS_COLLECTION: [reagent_groups_record]
    }
}
RAW_SNAPSHOT_PATH = PRIVATE_RUN_DIR / "pocketbase_snapshot.json"
RAW_SNAPSHOT_PATH.write_text(json.dumps(snapshot, ensure_ascii=False, indent=2), encoding="utf-8")

print(f"Fetched {designs_result['records_fetched']} design records.")
print(f"PocketBase advertised total: {designs_result['advertised_total_items']}.")
print(f"Possibly truncated: {designs_result['possibly_truncated']}.")
print(f"Raw snapshot stored outside Git at: {RAW_SNAPSHOT_PATH}")


In [ ]:
# 6. Parse the source-defined data shape without changing the raw snapshot.
import pandas as pd
import numpy as np
import hashlib


def parse_json_value(value):
    if isinstance(value, str):
        stripped = value.strip()
        if not stripped:
            return None
        try:
            return json.loads(stripped)
        except Exception:
            return value
    return value


def as_dict(value):
    value = parse_json_value(value)
    return value if isinstance(value, dict) else {}


def norm_target(value):
    if not value:
        return "Unknown"
    s = str(value).strip()
    key = s.casefold()
    known = {"sfgfp": "sfGFP", "sf gfp": "sfGFP", "petase": "PETase", "reteplase": "Reteplase"}
    return known.get(key, s)


def extract_design_row(rec):
    design = as_dict(rec.get("design_json"))
    metadata = as_dict(design.get("metadata"))
    reagent_list = as_dict(design.get("reagent_list"))
    sample_id = design.get("sample_id") or rec.get("title") or rec.get("id")
    batch_id = metadata.get("batch_id") or None
    return {
        "record_id": rec.get("id"),
        "sample_id": str(sample_id) if sample_id is not None else None,
        "created": rec.get("created"),
        "updated": rec.get("updated"),
        "protein_target": norm_target(metadata.get("protein_target")),
        "sample_type": design.get("sample_type"),
        "batch_id": str(batch_id) if batch_id else None,
        "batch_size": pd.to_numeric(metadata.get("batch_size"), errors="coerce"),
        "batch_index": pd.to_numeric(metadata.get("batch_index"), errors="coerce"),
        "has_experiment_hypothesis": bool(str(metadata.get("experiment_hypothesis") or "").strip()),
        "has_comments": bool(str(metadata.get("comments") or rec.get("rationale") or "").strip()),
        "has_reagent_list": bool(reagent_list),
        "reagent_list": reagent_list,
        # Keep text fields in memory for internal analysis, but never export them to reports.
        "experiment_hypothesis_private": str(metadata.get("experiment_hypothesis") or ""),
        "comments_private": str(metadata.get("comments") or rec.get("rationale") or ""),
        "reagent_list_json": json.dumps(reagent_list, sort_keys=True, separators=(",", ":")),
    }

normalized_rows = [extract_design_row(rec) for rec in snapshot["collections"][DESIGNS_COLLECTION]]
designs_df = pd.DataFrame(normalized_rows)
if designs_df.empty:
    raise RuntimeError("No cfps_designs records were returned. Check access and the source schema before continuing.")

def explode_reagents(df):
    out = []
    for row in df.to_dict(orient="records"):
        rlist = row.get("reagent_list") if isinstance(row.get("reagent_list"), dict) else {}
        for reagent_id, raw_volume in rlist.items():
            value = pd.to_numeric(raw_volume, errors="coerce")
            out.append({
                "record_id": row.get("record_id"),
                "sample_id": row.get("sample_id"),
                "protein_target": row.get("protein_target"),
                "batch_id": row.get("batch_id"),
                "reagent_id": str(reagent_id),
                "volume_nL_as_stored": float(value) if pd.notna(value) else np.nan,
            })
    return pd.DataFrame(out, columns=["record_id", "sample_id", "protein_target", "batch_id", "reagent_id", "volume_nL_as_stored"])

reagent_volumes_df = explode_reagents(designs_df)

def formulation_hash(reagent_list_json, protein_target):
    payload = f"{protein_target}|{reagent_list_json}".encode("utf-8")
    return hashlib.sha256(payload).hexdigest()

designs_df["formulation_hash"] = designs_df.apply(
    lambda row: formulation_hash(row["reagent_list_json"], row["protein_target"]), axis=1
)

print(f"Normalized designs: {len(designs_df):,}")
print(f"Rows with a usable reagent_list: {int(designs_df['has_reagent_list'].sum()):,}")
print(f"Long-form reagent entries: {len(reagent_volumes_df):,}")
print("Targets (including Unknown):")
display(designs_df["protein_target"].value_counts(dropna=False).rename_axis("protein_target").to_frame("n_designs"))


In [ ]:
# 7. Parse the source-confirmed reagent catalogue for labels and context (private working table).
# The source normalizer supports a grouped `reagents` array; keep unknown shapes visible.
reagent_record = snapshot["collections"][REAGENT_GROUPS_COLLECTION][0]

def parse_catalog_groups(record):
    candidates = [
        record.get("reagentGroups"), record.get("reagent_groups"), record.get("groups"),
        record.get("data"), record.get("json"), record.get("payload"), record.get("content"),
        record.get("reagents")
    ]
    parsed_candidates = [parse_json_value(x) for x in candidates]
    for candidate in parsed_candidates:
        if isinstance(candidate, list):
            rows = []
            for group in candidate:
                if not isinstance(group, dict):
                    continue
                group_name = group.get("name") or group.get("group") or group.get("group_name") or "Uncategorized"
                items = parse_json_value(group.get("reagents")) or parse_json_value(group.get("items")) or []
                if isinstance(items, list):
                    for item in items:
                        if not isinstance(item, dict):
                            continue
                        rid = item.get("id") or item.get("reagent_id") or item.get("slug") or item.get("key")
                        if not rid:
                            continue
                        rows.append({
                            "group": group_name,
                            "reagent_id": str(rid),
                            "name": item.get("name") or item.get("reagent_name") or item.get("label") or str(rid),
                            "concentration_or_units": item.get("concentration") or item.get("units"),
                            "cost_per_ml": item.get("costPerMl", item.get("cost_per_ml", item.get("cost_per_mL"))),
                            "in_stock": item.get("in_stock", item.get("inStock")),
                            "hidden": item.get("hidden", False),
                        })
            if rows:
                return rows
    return []

catalog_rows = parse_catalog_groups(reagent_record)
reagent_catalog_df = pd.DataFrame(catalog_rows)
if reagent_catalog_df.empty:
    print("WARNING: No catalog rows recognized. Raw catalogue stays in the private snapshot; review the actual record shape before defining labels.")
else:
    print(f"Parsed {len(reagent_catalog_df)} reagent-catalogue entries from the source record.")
    display(reagent_catalog_df.head(10))


In [ ]:
# 8. Compute descriptive QC tables. These are aggregate/derived outputs suitable for review;
# raw records, author fields, conversation links, comments, and row-level formulation exports
# are deliberately not written into the repository reports directory.

# Counts per target.
counts_by_target = (
    designs_df.groupby("protein_target", dropna=False)
    .agg(
        n_designs=("record_id", "nunique"),
        n_with_reagent_list=("has_reagent_list", "sum"),
        n_with_hypothesis=("has_experiment_hypothesis", "sum"),
        n_with_comments=("has_comments", "sum"),
        n_with_batch_id=("batch_id", lambda s: s.notna().sum()),
    )
    .reset_index()
    .sort_values("protein_target")
)

# Date counts; only date and target counts are exported.
designs_df["created_utc"] = pd.to_datetime(designs_df["created"], errors="coerce", utc=True)
over_time = (
    designs_df.dropna(subset=["created_utc"])
    .assign(date=lambda x: x["created_utc"].dt.strftime("%Y-%m-%d"))
    .groupby(["date", "protein_target"], dropna=False)
    .size().rename("n_designs").reset_index()
    .sort_values(["date", "protein_target"])
)

# Formulation duplicate summary: repeated recipes are counted within a target.
duplicate_groups = (
    designs_df.groupby(["protein_target", "formulation_hash"], dropna=False)
    .size().rename("n_records").reset_index()
)
duplicate_summary = (
    duplicate_groups.assign(is_duplicate=lambda x: x["n_records"] > 1)
    .groupby("protein_target", dropna=False)
    .agg(
        n_unique_formulations=("formulation_hash", "nunique"),
        n_duplicate_recipe_groups=("is_duplicate", "sum"),
        records_in_duplicate_groups=("n_records", lambda s: int(s[s > 1].sum())),
    )
    .reset_index()
)

# Reaction-total diagnostic. Repository source documents a 20,000 nL reaction as the current contract.
# This is a warning table, not a filter: legacy designs may follow different assumptions.
volume_totals = []
for row in designs_df.to_dict(orient="records"):
    rlist = row.get("reagent_list") if isinstance(row.get("reagent_list"), dict) else {}
    numeric = [pd.to_numeric(v, errors="coerce") for v in rlist.values()]
    numeric = [float(v) for v in numeric if pd.notna(v)]
    total = sum(numeric) if numeric else np.nan
    volume_totals.append({
        "record_id": row.get("record_id"),
        "protein_target": row.get("protein_target"),
        "total_volume_nL": total,
        "matches_20000_nL": bool(pd.notna(total) and abs(total - 20000) < 1e-6),
        "has_numeric_volumes": bool(numeric),
    })
volume_qc_df = pd.DataFrame(volume_totals)
volume_qc_summary = (
    volume_qc_df.groupby("protein_target", dropna=False)
    .agg(
        n_designs=("record_id", "nunique"),
        n_with_numeric_volumes=("has_numeric_volumes", "sum"),
        n_matching_20000_nL=("matches_20000_nL", "sum"),
        median_total_volume_nL=("total_volume_nL", "median"),
    ).reset_index()
)

print("Counts by target")
display(counts_by_target)
print("Duplicate-formulation summary")
display(duplicate_summary)
print("20,000 nL QC diagnostic (mismatches are not automatically invalid)")
display(volume_qc_summary)


In [ ]:
# 9. Prepare a coverage matrix: percent of designs per target with each ingredient > 0 nL.
# Missing fields are treated as absent for this descriptive chart, matching the app's sparse reagent_list shape.
# Volumes remain `volume_nL_as_stored`; no concentration conversion is attempted.

coverage_rows = []
if not reagent_volumes_df.empty:
    valid_long = reagent_volumes_df.dropna(subset=["volume_nL_as_stored"]).copy()
    positive = valid_long[valid_long["volume_nL_as_stored"] > 0]
    n_per_target = designs_df.groupby("protein_target")["record_id"].nunique().to_dict()
    reagent_counts = positive.groupby(["protein_target", "reagent_id"])["record_id"].nunique().reset_index(name="n_designs_with_reagent")
    for r in reagent_counts.to_dict(orient="records"):
        denom = int(n_per_target.get(r["protein_target"], 0))
        coverage_rows.append({
            "protein_target": r["protein_target"],
            "reagent_id": r["reagent_id"],
            "n_designs_with_reagent": int(r["n_designs_with_reagent"]),
            "n_designs_in_target": denom,
            "prevalence_pct": (100 * r["n_designs_with_reagent"] / denom) if denom else np.nan,
        })
coverage_df = pd.DataFrame(coverage_rows)
if coverage_df.empty:
    print("No numeric reagent-volume data available for coverage analysis.")
else:
    display(coverage_df.sort_values(["protein_target", "prevalence_pct"], ascending=[True, False]).head(20))


In [ ]:
# 10. Create publication-friendly exploratory figures using aggregate data only.
# The figure directory is in the repository; the private data tables are not.
import matplotlib.pyplot as plt

FIGURE_DIR = REPORT_DIR / "figures"
FIGURE_DIR.mkdir(parents=True, exist_ok=True)

# A) Count of formulation records per target.
fig, ax = plt.subplots(figsize=(7.5, 4.5))
plot_counts = counts_by_target.sort_values("n_designs", ascending=False)
ax.bar(plot_counts["protein_target"].astype(str), plot_counts["n_designs"])
ax.set_title("CFPS formulation records by target")
ax.set_xlabel("Target protein")
ax.set_ylabel("Number of records")
ax.tick_params(axis="x", rotation=20)
fig.tight_layout()
fig.savefig(FIGURE_DIR / "submissions_by_target.png", dpi=220, bbox_inches="tight")
plt.show()
plt.close(fig)

# B) Daily submission counts, if creation timestamps are available.
if not over_time.empty:
    date_pivot = over_time.pivot(index="date", columns="protein_target", values="n_designs").fillna(0).sort_index()
    fig, ax = plt.subplots(figsize=(10, 5))
    for col in date_pivot.columns:
        ax.plot(date_pivot.index, date_pivot[col], marker="o", linewidth=1.5, label=str(col))
    ax.set_title("CFPS formulation records over time")
    ax.set_xlabel("Creation date (UTC)")
    ax.set_ylabel("Records per day")
    ax.tick_params(axis="x", rotation=45)
    ax.legend(title="Target", frameon=False)
    fig.tight_layout()
    fig.savefig(FIGURE_DIR / "submissions_over_time.png", dpi=220, bbox_inches="tight")
    plt.show()
    plt.close(fig)
else:
    print("Skipped timeline figure: no valid created timestamps were found.")

# C) Ingredient prevalence by target; show the 20 most commonly used ingredients overall.
if not coverage_df.empty:
    top_reagents = (
        coverage_df.groupby("reagent_id")["n_designs_with_reagent"].sum()
        .sort_values(ascending=False).head(20).index.tolist()
    )
    cov = coverage_df[coverage_df["reagent_id"].isin(top_reagents)].pivot(
        index="reagent_id", columns="protein_target", values="prevalence_pct"
    )
    cov = cov.reindex(top_reagents)
    fig, ax = plt.subplots(figsize=(10, max(5, 0.34 * len(cov))))
    image = ax.imshow(cov.to_numpy(dtype=float), aspect="auto", interpolation="nearest", vmin=0, vmax=100)
    ax.set_title("Ingredient prevalence in formulations (top 20 ingredients)")
    ax.set_xlabel("Target protein")
    ax.set_ylabel("Reagent ID")
    ax.set_xticks(range(len(cov.columns)), labels=[str(c) for c in cov.columns], rotation=20, ha="right")
    ax.set_yticks(range(len(cov.index)), labels=[str(x) for x in cov.index])
    colorbar = fig.colorbar(image, ax=ax)
    colorbar.set_label("Designs containing reagent (%)")
    fig.tight_layout()
    fig.savefig(FIGURE_DIR / "reagent_prevalence_by_target.png", dpi=220, bbox_inches="tight")
    plt.show()
    plt.close(fig)
else:
    print("Skipped reagent prevalence figure: no numeric reagent volumes were recognized.")

print(f"Aggregate figures saved under {FIGURE_DIR}")


In [ ]:
# 11. Save aggregate reports inside the repo and the detailed, row-level working tables outside it.
# The public/reviewable reports deliberately exclude author, author_sub, comments, hypotheses,
# conversation links, sample IDs, and row-level recipe values.

# Aggregate tables (appropriate for coordinator review; still review before publishing).
counts_by_target.to_csv(REPORT_DIR / "counts_by_target.csv", index=False)
duplicate_summary.to_csv(REPORT_DIR / "duplicate_formulation_summary.csv", index=False)
volume_qc_summary.to_csv(REPORT_DIR / "volume_total_qc_by_target.csv", index=False)
if not over_time.empty:
    over_time.to_csv(REPORT_DIR / "daily_counts_by_target.csv", index=False)
if not coverage_df.empty:
    coverage_df.to_csv(REPORT_DIR / "reagent_coverage_by_target.csv", index=False)

# Detailed intermediate tables stay outside the repo for internal analysis only.
designs_private = designs_df.drop(columns=["created_utc"], errors="ignore").copy()
designs_private["reagent_list"] = designs_private["reagent_list"].apply(lambda x: json.dumps(x, ensure_ascii=False, sort_keys=True) if isinstance(x, dict) else "{}")
designs_private.to_csv(PRIVATE_RUN_DIR / "designs_normalized_private.csv", index=False)
reagent_volumes_df.to_csv(PRIVATE_RUN_DIR / "reagent_volumes_long_private.csv", index=False)
if not reagent_catalog_df.empty:
    reagent_catalog_df.to_csv(PRIVATE_RUN_DIR / "reagent_catalog_private.csv", index=False)

manifest = {
    "created_utc": datetime.now(timezone.utc).isoformat(),
    "source_repository": REPO_URL,
    "source_repository_commit": REPO_SHA,
    "pocketbase_host": PB_URL,
    "auth_collection": PB_AUTH_COLLECTION,
    "collections": {
        DESIGNS_COLLECTION: {
            "fetched": designs_result["records_fetched"],
            "advertised_total_items": designs_result["advertised_total_items"],
            "possibly_truncated": designs_result["possibly_truncated"],
        },
        REAGENT_GROUPS_COLLECTION: {"record_id": REAGENT_GROUPS_RECORD_ID, "fetched": 1},
    },
    "derived_outputs": [
        "counts_by_target.csv",
        "duplicate_formulation_summary.csv",
        "volume_total_qc_by_target.csv",
        "daily_counts_by_target.csv" if not over_time.empty else None,
        "reagent_coverage_by_target.csv" if not coverage_df.empty else None,
        "figures/submissions_by_target.png",
        "figures/submissions_over_time.png" if not over_time.empty else None,
        "figures/reagent_prevalence_by_target.png" if not coverage_df.empty else None,
    ],
    "raw_data_stored_outside_repository": True,
    "publication_caveat": "Aggregate outputs only; coordinator approval and manual privacy review required before publication."
}
(REPORT_DIR / "manifest.json").write_text(json.dumps(manifest, indent=2), encoding="utf-8")

print("Aggregate/review outputs:", REPORT_DIR)
print("Private detailed tables:", PRIVATE_RUN_DIR)
print("Reports saved:")
for p in sorted(REPORT_DIR.rglob("*")):
    if p.is_file():
        print(" -", p.relative_to(REPO_ROOT))


## 12. Pairwise correlation-based clustering of submissions

### What is being correlated?

This first implementation treats each CFPS submission as a vector whose features are reagent IDs and whose values are submitted reagent volumes. A pairwise correlation compares **two submission vectors across reagent features**. It measures similarity of recipe patterns; it is **not** a correlation between experimental yields, not a causal effect, and not expected information gain. The submission JSON described in the agent instructions carries `sample_id`, `sample_type`, `reagent_list` (reagent → nL), and `metadata`; in the PocketBase snapshot this object is nested under each record's `design_json`.

The reusable code is kept outside the notebook in `analysis/cfps_correlation/pairwise_correlation_clustering.py`. The abstract base class owns shared validation, correlations, distances, hierarchical clustering, exports, and plotting; the CFPS adapter turns the source-specific JSON into generic numeric feature vectors and aligned metadata.

**Defaults and cautions:** separate clustering by `protein_target`; Spearman correlation; average linkage; six clusters per target (configurable); remove features constant within a target group; use `1 − r` as a dissimilarity. Ward linkage is deliberately not allowed because `1 − r` is not guaranteed to be Euclidean. Missing reagent keys mean zero volume under the sparse submission contract, but malformed explicit values raise an error. These are recipe-pattern clusters, not validated function/performance groups.

The code-writing cells below materialize the reusable module and unit tests into the cloned repository checkout. Review the generated diff before committing. The raw pairwise matrices, row-level labels, and figures generated from live community submissions remain in `/content/cfps_private_snapshots/` by default.


In [ ]:
# 12.1 Prepare an analysis package location and ensure the output remains inside the cloned checkout.
from pathlib import Path

CLUSTERING_PACKAGE_DIR = REPO_ROOT / "analysis" / "cfps_correlation"
CLUSTERING_PACKAGE_DIR.mkdir(parents=True, exist_ok=True)
analysis_root = REPO_ROOT / "analysis"
analysis_root.mkdir(parents=True, exist_ok=True)
# Preserve an existing analysis package initializer if the repository already has one.
analysis_init = analysis_root / "__init__.py"
if not analysis_init.exists():
    analysis_init.write_text(
        '"""Analysis modules for reproducible HTGAA CFPS research workflows."""\n',
        encoding="utf-8",
    )
(REPO_ROOT / "tests").mkdir(parents=True, exist_ok=True)
print("Reusable package destination:", CLUSTERING_PACKAGE_DIR)
print("Raw clustering outputs will be kept outside the repository by default:", PRIVATE_RUN_DIR)


In [ ]:
%%writefile /content/htgaa-automation/analysis/cfps_correlation/__init__.py
"""Reusable CFPS-oriented correlation clustering tools."""
from .pairwise_correlation_clustering import (
    AbstractPairwiseCorrelationClusterer,
    CFPSSubmissionCorrelationAdapter,
    CorrelationClusterResult,
    adjusted_rand_index,
)

__all__ = [
    "AbstractPairwiseCorrelationClusterer",
    "CFPSSubmissionCorrelationAdapter",
    "CorrelationClusterResult",
    "adjusted_rand_index",
]


In [ ]:
%%writefile /content/htgaa-automation/analysis/cfps_correlation/pairwise_correlation_clustering.py
"""Reusable correlation-based clustering for entity-by-feature data.

The abstract class implements shared validation, pairwise correlation, conversion to
correlation dissimilarity, hierarchical clustering, output tables, and plots. A
subclass/adapter is responsible for converting a project-specific source (e.g. CFPS
submission JSON) into a numeric feature matrix and aligned metadata.

Interpretation warning
----------------------
For CFPS designs, the rows are submissions and the columns are reagent features.
The correlation between two rows measures similarity of their across-reagent *patterns*.
It does NOT measure correlation between experimental outcomes, causal relationships,
or expected information gain.
"""
from __future__ import annotations

from abc import ABC, abstractmethod
from dataclasses import dataclass
from pathlib import Path
import re
from typing import Any, Mapping, Optional

import numpy as np
import pandas as pd


def adjusted_rand_index(labels_a: Any, labels_b: Any) -> float:
    """Compute adjusted Rand index for two partitions over the same entities.

    Labels are aligned by position; callers must align entity IDs first. The score
    is invariant to arbitrary cluster label names and is intended for sensitivity
    analysis, not for choosing a preferred clustering by itself.
    """
    a = np.asarray(labels_a, dtype=object).ravel()
    b = np.asarray(labels_b, dtype=object).ravel()
    if len(a) != len(b):
        raise ValueError("Both label arrays must have the same length")
    n = len(a)
    if n < 2:
        return float("nan")
    contingency = pd.crosstab(pd.Series(a, name="a"), pd.Series(b, name="b")).to_numpy(dtype=float)
    choose2 = lambda x: x * (x - 1.0) / 2.0
    sum_cells = float(choose2(contingency).sum())
    sum_rows = float(choose2(contingency.sum(axis=1)).sum())
    sum_cols = float(choose2(contingency.sum(axis=0)).sum())
    total_pairs = float(choose2(n))
    expected = (sum_rows * sum_cols) / total_pairs
    maximum = 0.5 * (sum_rows + sum_cols)
    denominator = maximum - expected
    if abs(denominator) < np.finfo(float).eps:
        # Degenerate partitions: score identical pairwise memberships as 1, else 0.
        same = all((a[i] == a[j]) == (b[i] == b[j]) for i in range(n) for j in range(i + 1, n))
        return 1.0 if same else 0.0
    return float((sum_cells - expected) / denominator)


@dataclass
class CorrelationClusterResult:
    """Results for a single group of entities (e.g. one protein target)."""

    group_name: str
    feature_matrix: pd.DataFrame
    correlation_matrix: pd.DataFrame
    distance_matrix: pd.DataFrame
    labels: pd.Series
    linkage_matrix: np.ndarray
    leaf_order: list[str]
    dropped_entities: list[str]
    dropped_features: list[str]
    correlation_method: str
    linkage_method: str
    feature_mode: str


class AbstractPairwiseCorrelationClusterer(ABC):
    """Reusable analysis workflow for correlation-based hierarchical clustering.

    Subclasses must implement ``build_feature_matrix`` and ``build_metadata``.
    Feature matrix rows are entities and columns are numeric features. Entity indexes
    must be unique strings and must align with the metadata index. Missing feature
    values are rejected in the abstract layer: each adapter must state how its source
    encodes absence/missingness rather than having zeros guessed silently.

    ``1 - correlation`` is used as a dissimilarity. Since it is not guaranteed to be
    Euclidean, Ward/centroid/median linkage are intentionally disallowed; average or
    complete linkage are reasonable defaults for this distance.
    """

    ALLOWED_CORRELATIONS = {"pearson", "spearman"}
    ALLOWED_LINKAGES = {"average", "complete", "single", "weighted"}

    def __init__(
        self,
        *,
        correlation_method: str = "spearman",
        linkage_method: str = "average",
        n_clusters: int = 6,
        min_features: int = 3,
        group_by: Optional[str] = "protein_target",
    ) -> None:
        self.correlation_method = correlation_method.lower()
        self.linkage_method = linkage_method.lower()
        self.n_clusters = int(n_clusters)
        self.min_features = int(min_features)
        self.group_by = group_by

        if self.correlation_method not in self.ALLOWED_CORRELATIONS:
            raise ValueError(f"correlation_method must be one of {sorted(self.ALLOWED_CORRELATIONS)}")
        if self.linkage_method not in self.ALLOWED_LINKAGES:
            raise ValueError(
                f"linkage_method must be one of {sorted(self.ALLOWED_LINKAGES)}; "
                "Ward/centroid/median are not supported for 1-r dissimilarity."
            )
        if self.n_clusters < 1:
            raise ValueError("n_clusters must be >= 1")
        if self.min_features < 2:
            raise ValueError("min_features must be >= 2")

        self.results_: dict[str, CorrelationClusterResult] = {}
        self.skipped_groups_: dict[str, str] = {}
        self.feature_matrix_: Optional[pd.DataFrame] = None
        self.metadata_: Optional[pd.DataFrame] = None

    @abstractmethod
    def build_feature_matrix(self) -> pd.DataFrame:
        """Return a numeric DataFrame: entities as rows, features as columns."""
        raise NotImplementedError

    @abstractmethod
    def build_metadata(self) -> pd.DataFrame:
        """Return entity metadata indexed by the same entity IDs as the feature matrix."""
        raise NotImplementedError

    @staticmethod
    def _safe_group_name(value: Any) -> str:
        if value is None or (isinstance(value, float) and np.isnan(value)):
            return "Unknown"
        text = str(value).strip()
        return text if text else "Unknown"

    @staticmethod
    def _slug(value: str) -> str:
        slug = re.sub(r"[^A-Za-z0-9._-]+", "_", str(value)).strip("_.-")
        return slug or "group"

    def validate_inputs(self) -> tuple[pd.DataFrame, pd.DataFrame]:
        """Build and validate matrices; does not mutate the adapter's source records."""
        X = self.build_feature_matrix().copy()
        metadata = self.build_metadata().copy()
        if not isinstance(X, pd.DataFrame) or X.empty:
            raise ValueError("build_feature_matrix() must return a non-empty pandas DataFrame")
        if not isinstance(metadata, pd.DataFrame):
            raise TypeError("build_metadata() must return a pandas DataFrame")

        X.index = X.index.map(str)
        metadata.index = metadata.index.map(str)
        if X.index.has_duplicates:
            raise ValueError("Feature matrix index must be unique; create stable unique entity IDs in the adapter")
        if metadata.index.has_duplicates:
            raise ValueError("Metadata index must be unique")
        if not X.index.equals(metadata.index):
            missing = sorted(set(X.index) - set(metadata.index))[:5]
            extra = sorted(set(metadata.index) - set(X.index))[:5]
            raise ValueError(f"Feature/metadata indexes must match and be ordered alike. Missing metadata={missing}; extra metadata={extra}")

        X = X.apply(pd.to_numeric, errors="coerce")
        if X.columns.duplicated().any():
            raise ValueError("Feature names must be unique")
        if X.isna().any().any():
            n = int(X.isna().sum().sum())
            raise ValueError(
                f"Feature matrix contains {n} missing/non-numeric values. Resolve missingness explicitly in the adapter; "
                "do not silently turn unknown values into zero."
            )
        values = X.to_numpy(dtype=float)
        if not np.isfinite(values).all():
            raise ValueError("Feature matrix contains infinite values")
        self.feature_matrix_ = X
        self.metadata_ = metadata
        return X, metadata

    def _fit_one_group(self, group_name: str, X: pd.DataFrame) -> Optional[CorrelationClusterResult]:
        if len(X) < 2:
            self.skipped_groups_[group_name] = f"Only {len(X)} entity/entities; need at least 2."
            return None

        # Invariant features carry no discriminatory information within this group.
        variable_mask = X.nunique(axis=0, dropna=False) > 1
        dropped_features = X.columns[~variable_mask].astype(str).tolist()
        Xv = X.loc[:, variable_mask].copy()
        if Xv.shape[1] < self.min_features:
            self.skipped_groups_[group_name] = (
                f"Only {Xv.shape[1]} variable feature(s) after removing constants; "
                f"need at least {self.min_features}."
            )
            return None

        # Correlation is undefined for an entity vector with zero variance across features.
        row_sd = Xv.std(axis=1, ddof=0)
        valid_rows = row_sd > np.finfo(float).eps
        dropped_entities = Xv.index[~valid_rows].astype(str).tolist()
        Xv = Xv.loc[valid_rows]
        if len(Xv) < 2:
            self.skipped_groups_[group_name] = "Fewer than two non-constant entity vectors remain."
            return None

        correlation = Xv.T.corr(method=self.correlation_method, min_periods=self.min_features)
        correlation = correlation.clip(lower=-1.0, upper=1.0)
        correlation = (correlation + correlation.T) / 2.0
        np.fill_diagonal(correlation.values, 1.0)
        if not np.isfinite(correlation.to_numpy(dtype=float)).all():
            raise ValueError(
                f"Group {group_name!r} produced undefined pairwise correlations. "
                "Check sparse/tied features or raise feature quality before clustering."
            )

        distance = (1.0 - correlation).clip(lower=0.0, upper=2.0)
        distance = (distance + distance.T) / 2.0
        np.fill_diagonal(distance.values, 0.0)

        from scipy.cluster.hierarchy import linkage, fcluster, leaves_list
        from scipy.spatial.distance import squareform

        condensed = squareform(distance.to_numpy(dtype=float), checks=True)
        Z = linkage(condensed, method=self.linkage_method, optimal_ordering=True)
        k = min(self.n_clusters, len(Xv))
        cluster_numbers = fcluster(Z, t=k, criterion="maxclust")
        labels = pd.Series(
            [f"C{int(v):02d}" for v in cluster_numbers], index=Xv.index, name="cluster_id"
        )
        order = leaves_list(Z)
        leaf_order = Xv.index[order].astype(str).tolist()

        return CorrelationClusterResult(
            group_name=group_name,
            feature_matrix=Xv,
            correlation_matrix=correlation,
            distance_matrix=distance,
            labels=labels,
            linkage_matrix=Z,
            leaf_order=leaf_order,
            dropped_entities=dropped_entities,
            dropped_features=dropped_features,
            correlation_method=self.correlation_method,
            linkage_method=self.linkage_method,
            feature_mode=getattr(self, "feature_mode", "adapter-defined"),
        )

    def fit(self) -> "AbstractPairwiseCorrelationClusterer":
        """Run one clustering per group (protein target by default)."""
        X, metadata = self.validate_inputs()
        self.results_ = {}
        self.skipped_groups_ = {}

        if self.group_by and self.group_by in metadata.columns:
            groups = metadata[self.group_by].map(self._safe_group_name)
            grouped_indices = [(name, metadata.index[groups == name]) for name in pd.unique(groups)]
        elif self.group_by is None:
            grouped_indices = [("All", metadata.index)]
        else:
            raise ValueError(f"group_by={self.group_by!r} is not a metadata column; available: {list(metadata.columns)}")

        for name, ids in grouped_indices:
            result = self._fit_one_group(str(name), X.loc[ids])
            if result is not None:
                self.results_[str(name)] = result

        if not self.results_:
            raise ValueError(f"No groups could be clustered. Skip reasons: {self.skipped_groups_}")
        return self

    def labels_frame(self) -> pd.DataFrame:
        """Return row-level labels plus metadata. Treat as private unless approved for sharing."""
        if not self.results_:
            raise RuntimeError("Call fit() before requesting labels")
        assert self.metadata_ is not None
        frames = []
        for group_name, result in self.results_.items():
            part = self.metadata_.loc[result.labels.index].copy()
            part.insert(0, "entity_id", part.index.astype(str))
            part["cluster_group"] = group_name
            part["cluster_id"] = result.labels.astype(str)
            part["correlation_method"] = result.correlation_method
            part["linkage_method"] = result.linkage_method
            part["feature_mode"] = result.feature_mode
            frames.append(part.reset_index(drop=True))
        return pd.concat(frames, ignore_index=True) if frames else pd.DataFrame()

    def compare_with(self, other: "AbstractPairwiseCorrelationClusterer") -> pd.DataFrame:
        """Compare this fit with another fit using adjusted Rand index by group.

        Only entities clustered by both runs are compared; the overlap count/fraction
        is reported so that an apparently good score based on a small subset is visible.
        """
        if not self.results_ or not other.results_:
            raise RuntimeError("Both clusterers must be fitted before compare_with()")
        rows = []
        for group in sorted(set(self.results_) & set(other.results_)):
            left = self.results_[group].labels.rename("left")
            right = other.results_[group].labels.rename("right")
            joined = pd.concat([left, right], axis=1, join="inner").dropna()
            n_left = len(left)
            n_right = len(right)
            rows.append({
                "group": group,
                "n_overlap": int(len(joined)),
                "n_left": int(n_left),
                "n_right": int(n_right),
                "overlap_pct_of_left": (100.0 * len(joined) / n_left) if n_left else np.nan,
                "overlap_pct_of_right": (100.0 * len(joined) / n_right) if n_right else np.nan,
                "adjusted_rand_index": adjusted_rand_index(joined["left"], joined["right"]) if len(joined) >= 2 else np.nan,
            })
        return pd.DataFrame(rows, columns=["group", "n_overlap", "n_left", "n_right", "overlap_pct_of_left", "overlap_pct_of_right", "adjusted_rand_index"])

    def cluster_sizes_frame(self) -> pd.DataFrame:
        """Aggregate cluster sizes; review before publication because groups can be small."""
        rows = []
        for group_name, result in self.results_.items():
            counts = result.labels.value_counts().sort_index()
            for cluster_id, count in counts.items():
                rows.append({"group": group_name, "cluster_id": cluster_id, "n_entities": int(count)})
        return pd.DataFrame(rows, columns=["group", "cluster_id", "n_entities"])

    def save_private_outputs(self, output_dir: str | Path) -> list[Path]:
        """Write row-level matrices/labels to a caller-specified private directory."""
        if not self.results_:
            raise RuntimeError("Call fit() before saving outputs")
        out = Path(output_dir)
        out.mkdir(parents=True, exist_ok=True)
        written: list[Path] = []
        label_path = out / "cluster_labels_private.csv"
        self.labels_frame().to_csv(label_path, index=False)
        written.append(label_path)
        for name, result in self.results_.items():
            slug = self._slug(name)
            for filename, table in (
                (f"correlation_{slug}_private.csv", result.correlation_matrix),
                (f"distance_{slug}_private.csv", result.distance_matrix),
                (f"features_{slug}_private.csv", result.feature_matrix),
            ):
                path = out / filename
                table.to_csv(path, index=True)
                written.append(path)
            path = out / f"cluster_sizes_{slug}_aggregate.csv"
            result.labels.value_counts().rename_axis("cluster_id").rename("n_entities").reset_index().to_csv(path, index=False)
            written.append(path)
        return written

    def plot_correlation_heatmap(
        self,
        group_name: str,
        *,
        annotate_max_entities: int = 35,
        figsize: Optional[tuple[float, float]] = None,
        save_path: str | Path | None = None,
    ):
        """Plot pairwise correlations, ordered by hierarchical leaves and cluster."""
        import matplotlib.pyplot as plt

        result = self.results_[group_name]
        order = result.leaf_order
        corr = result.correlation_matrix.loc[order, order]
        n = len(order)
        if figsize is None:
            side = min(16.0, max(6.0, n * 0.035))
            figsize = (side, side)
        fig, ax = plt.subplots(figsize=figsize)
        image = ax.imshow(corr.to_numpy(dtype=float), vmin=-1, vmax=1, cmap="coolwarm", interpolation="nearest", aspect="auto")
        cb = fig.colorbar(image, ax=ax, fraction=0.046, pad=0.04)
        cb.set_label(f"{result.correlation_method.title()} correlation (recipe vectors)")
        ax.set_title(f"Pairwise recipe correlation — {group_name}\n{n} submissions · {result.linkage_method} linkage · {result.feature_mode} features")
        if n <= annotate_max_entities:
            ax.set_xticks(range(n), labels=order, rotation=90, fontsize=7)
            ax.set_yticks(range(n), labels=order, fontsize=7)
        else:
            ax.set_xticks([])
            ax.set_yticks([])
        # Draw separators between adjacent clusters in leaf order.
        ordered_labels = result.labels.loc[order].astype(str).tolist()
        for i in range(1, n):
            if ordered_labels[i] != ordered_labels[i - 1]:
                ax.axhline(i - 0.5, color="black", linewidth=0.7, alpha=0.7)
                ax.axvline(i - 0.5, color="black", linewidth=0.7, alpha=0.7)
        fig.tight_layout()
        if save_path is not None:
            Path(save_path).parent.mkdir(parents=True, exist_ok=True)
            fig.savefig(save_path, dpi=220, bbox_inches="tight")
        return fig, ax

    def plot_dendrogram(
        self,
        group_name: str,
        *,
        show_labels_max_entities: int = 60,
        figsize: Optional[tuple[float, float]] = None,
        save_path: str | Path | None = None,
    ):
        """Plot a hierarchical dendrogram; suppress labels on large groups."""
        import matplotlib.pyplot as plt
        from scipy.cluster.hierarchy import dendrogram

        result = self.results_[group_name]
        n = len(result.labels)
        if figsize is None:
            figsize = (max(10.0, min(22.0, n * 0.06)), 6.5)
        fig, ax = plt.subplots(figsize=figsize)
        dendrogram(
            result.linkage_matrix,
            labels=result.feature_matrix.index.astype(str).tolist() if n <= show_labels_max_entities else None,
            leaf_rotation=90,
            leaf_font_size=7,
            ax=ax,
            no_labels=(n > show_labels_max_entities),
            color_threshold=None,
        )
        ax.set_title(f"Hierarchical clustering — {group_name}")
        ax.set_xlabel("Submission" if n <= show_labels_max_entities else "Submissions (labels suppressed)")
        ax.set_ylabel("1 − {0} correlation dissimilarity".format(result.correlation_method))
        fig.tight_layout()
        if save_path is not None:
            Path(save_path).parent.mkdir(parents=True, exist_ok=True)
            fig.savefig(save_path, dpi=220, bbox_inches="tight")
        return fig, ax

    def plot_cluster_sizes(
        self,
        group_name: str,
        *,
        figsize: tuple[float, float] = (7.5, 4.5),
        save_path: str | Path | None = None,
    ):
        """Plot the number of entities in each cluster."""
        import matplotlib.pyplot as plt

        result = self.results_[group_name]
        counts = result.labels.value_counts().sort_index()
        fig, ax = plt.subplots(figsize=figsize)
        ax.bar(counts.index.astype(str), counts.values)
        ax.set_title(f"Cluster sizes — {group_name}")
        ax.set_xlabel("Cluster")
        ax.set_ylabel("Number of submissions")
        fig.tight_layout()
        if save_path is not None:
            Path(save_path).parent.mkdir(parents=True, exist_ok=True)
            fig.savefig(save_path, dpi=220, bbox_inches="tight")
        return fig, ax


class CFPSSubmissionCorrelationAdapter(AbstractPairwiseCorrelationClusterer):
    """Adapter from the CFPS coopetition snapshot's records to recipe feature vectors.

    Accepted input shapes:
      * raw ``cfps_designs`` record list;
      * snapshot mapping with ``collections.cfps_designs``;
      * iterable PocketBase records with ``design_json`` as a dict or JSON string;
      * a single submission payload (``{"sample": ...}``) or batch payload
        (``{"samples": [...]}``) matching the coopetition agent instructions.

    In the agent/API contract, a sample has ``sample_id``, ``sample_type``,
    ``reagent_list`` (reagent ID -> volume in nL), and ``metadata``; the live PocketBase
    collection wraps this under ``design_json``. Omitted reagent keys are treated as
    zero volume because the submission contract uses a sparse reagent mapping. A key
    present with a non-numeric value is not silently treated as zero; that row's feature
    value is marked invalid and input validation will fail loudly.

    ``feature_mode`` options:
      * ``volume``: recorded nL volumes (default);
      * ``log1p_volume``: log(1 + nL), to reduce domination by large volume values;
      * ``presence``: 1 if volume > 0, otherwise 0.
    """

    FEATURE_MODES = {"volume", "log1p_volume", "presence"}

    def __init__(
        self,
        source: Any,
        *,
        feature_mode: str = "volume",
        targets: Optional[list[str]] = None,
        **kwargs: Any,
    ) -> None:
        super().__init__(**kwargs)
        if feature_mode not in self.FEATURE_MODES:
            raise ValueError(f"feature_mode must be one of {sorted(self.FEATURE_MODES)}")
        self.feature_mode = feature_mode
        self.targets = set(targets) if targets is not None else None
        self.records = self._extract_records(source)
        self._parsed_rows = self._parse_records(self.records)
        if self.targets is not None:
            allowed = {self._normalize_target(t) for t in self.targets}
            self._parsed_rows = [r for r in self._parsed_rows if r["protein_target"] in allowed]
        if not self._parsed_rows:
            raise ValueError("No CFPS submission records remain after parsing/target filtering")

    @staticmethod
    def _parse_json(value: Any) -> Any:
        if isinstance(value, str):
            text = value.strip()
            if not text:
                return None
            try:
                import json
                return json.loads(text)
            except (TypeError, ValueError):
                return value
        return value

    @classmethod
    def _as_dict(cls, value: Any) -> dict:
        parsed = cls._parse_json(value)
        return parsed if isinstance(parsed, dict) else {}

    @classmethod
    def _extract_records(cls, source: Any) -> list[dict]:
        inherited_batch_id = None
        if isinstance(source, Mapping):
            collections = source.get("collections", source)
            if isinstance(collections, Mapping) and "cfps_designs" in collections:
                records = collections["cfps_designs"]
            elif isinstance(source.get("samples"), (list, tuple)):
                # Agent API batch payload: {experiment_hypothesis, batch_id, samples: [...]}
                records = source["samples"]
                inherited_batch_id = source.get("batch_id") or source.get("batchId")
            elif isinstance(source.get("sample"), Mapping):
                # Agent API single-sample payload: {sample: {...}}
                records = [source]
            elif isinstance(source.get("reagent_list"), Mapping):
                # Bare sample object: {sample_id, sample_type, reagent_list, metadata}
                records = [source]
            else:
                raise ValueError(
                    "Mapping input must be a snapshot containing collections['cfps_designs'], "
                    "an agent payload with 'sample'/'samples', or a bare sample object."
                )
        else:
            records = source
        if not isinstance(records, (list, tuple)):
            raise TypeError("source must be a list of records, a snapshot mapping, or an agent sample payload")
        output = []
        for rec in records:
            if not isinstance(rec, Mapping):
                continue
            item = dict(rec)
            if inherited_batch_id is not None:
                item.setdefault("batch_id", inherited_batch_id)
            output.append(item)
        return output

    @staticmethod
    def _normalize_target(value: Any) -> str:
        if value is None or not str(value).strip():
            return "Unknown"
        raw = str(value).strip()
        known = {"sfgfp": "sfGFP", "sf gfp": "sfGFP", "petase": "PETase", "reteplase": "Reteplase"}
        return known.get(raw.casefold(), raw)

    def _parse_records(self, records: list[dict]) -> list[dict]:
        parsed_rows = []
        seen_entity_ids: set[str] = set()
        for idx, record in enumerate(records):
            design = self._as_dict(record.get("design_json"))
            # Agent POST payloads wrap a single composition under `sample`; other exports
            # may keep its fields directly on the record.
            if not design and isinstance(record.get("sample"), Mapping):
                design = self._as_dict(record.get("sample"))
            if not design and isinstance(record.get("reagent_list"), Mapping):
                design = record
            metadata = self._as_dict(design.get("metadata"))
            raw_reagents = self._as_dict(design.get("reagent_list"))
            target_value = metadata.get("protein_target", design.get("protein_target", record.get("protein_target", record.get("proteinTarget"))))
            target = self._normalize_target(target_value)
            if not raw_reagents:
                continue
            base_id = record.get("id") or design.get("sample_id") or record.get("title") or f"row_{idx:06d}"
            entity_id = str(base_id)
            if entity_id in seen_entity_ids:
                entity_id = f"{entity_id}__row_{idx:06d}"
            seen_entity_ids.add(entity_id)

            values: dict[str, float] = {}
            bad_fields: list[str] = []
            for reagent_id, raw_value in raw_reagents.items():
                reagent = str(reagent_id)
                try:
                    value = float(raw_value)
                except (TypeError, ValueError):
                    bad_fields.append(reagent)
                    continue
                if not np.isfinite(value) or value < 0:
                    bad_fields.append(reagent)
                    continue
                values[reagent] = value
            if bad_fields:
                raise ValueError(
                    f"Record {entity_id!r} has invalid/nonnegative-volume violations in reagent field(s): {bad_fields[:8]}"
                )
            if not values:
                continue
            if self.feature_mode == "presence":
                values = {k: float(v > 0) for k, v in values.items()}
            elif self.feature_mode == "log1p_volume":
                values = {k: float(np.log1p(v)) for k, v in values.items()}

            parsed_rows.append({
                "entity_id": entity_id,
                "record_id": str(record.get("id") or entity_id),
                "sample_id": design.get("sample_id") or record.get("title"),
                "protein_target": target,
                "batch_id": metadata.get("batch_id") or record.get("batch_id") or record.get("batchId"),
                "sample_type": design.get("sample_type"),
                "created": record.get("created"),
                "reagents": values,
            })
        return parsed_rows

    def build_feature_matrix(self) -> pd.DataFrame:
        # Missing mapping keys represent unadded reagents and are therefore zero in this API contract.
        feature_names = sorted({name for row in self._parsed_rows for name in row["reagents"]})
        rows = []
        ids = []
        for row in self._parsed_rows:
            rows.append({name: row["reagents"].get(name, 0.0) for name in feature_names})
            ids.append(row["entity_id"])
        return pd.DataFrame(rows, index=pd.Index(ids, name="entity_id"), columns=feature_names, dtype=float)

    def build_metadata(self) -> pd.DataFrame:
        columns = ["record_id", "sample_id", "protein_target", "batch_id", "sample_type", "created"]
        rows = []
        ids = []
        for row in self._parsed_rows:
            ids.append(row["entity_id"])
            rows.append({key: row.get(key) for key in columns})
        return pd.DataFrame(rows, index=pd.Index(ids, name="entity_id"), columns=columns)

    def feature_prevalence(self) -> pd.DataFrame:
        """Return a private QC table showing how frequently each reagent is present."""
        X = self.build_feature_matrix()
        return pd.DataFrame({
            "reagent_id": X.columns.astype(str),
            "n_entities_present": (X > 0).sum(axis=0).to_numpy(dtype=int),
            "n_entities": len(X),
            "prevalence_pct": 100.0 * (X > 0).mean(axis=0).to_numpy(dtype=float),
        }).sort_values(["prevalence_pct", "reagent_id"], ascending=[False, True]).reset_index(drop=True)


In [ ]:
%%writefile /content/htgaa-automation/analysis/cfps_correlation/README.md
# CFPS pairwise correlation clustering

This module adds a reusable pairwise-correlation hierarchical clustering step to the HTGAA CFPS snapshot workflow. It is designed to be contributed to [`RonanChance/htgaa-automation`](https://github.com/RonanChance/htgaa-automation) after review.

## Important interpretation

The implementation treats each submission as a numeric vector over reagent features and computes pairwise Pearson or Spearman correlation **between submission vectors**. A correlation is therefore similarity between recipe patterns across reagent features. It is **not** correlation between measured yield/fitness outcomes, not a causal effect, and not expected information gain. Current submission JSON does not supply verified experimental outcome labels.

## Architecture

- `AbstractPairwiseCorrelationClusterer` defines the reusable interface and implements validation, invariant-feature removal, pairwise correlations, `1 - r` dissimilarities, hierarchical clustering, private CSV export, and figures (correlation heatmap, dendrogram, cluster-size plot).
- `CFPSSubmissionCorrelationAdapter` converts PocketBase `cfps_designs` records (`design_json` containing `sample_id`, `sample_type`, `reagent_list`, and `metadata`) or direct agent API single/batch submission payloads into one row per submission and one column per reagent.
- Missing reagent keys are treated as 0 nL because the submission contract represents a sparse reagent mapping. A present but invalid numeric value raises an error instead of being silently treated as zero.
- Submissions are clustered separately by `protein_target` by default, reducing the risk that target differences dominate within-target recipe relationships.

## Basic usage

```python
from analysis.cfps_correlation import CFPSSubmissionCorrelationAdapter

clusterer = CFPSSubmissionCorrelationAdapter(
    snapshot,                         # raw snapshot mapping OR cfps_designs list
    feature_mode="volume",            # volume | log1p_volume | presence
    correlation_method="spearman",   # spearman | pearson
    linkage_method="average",         # average | complete | single | weighted
    n_clusters=6,
    min_features=3,
    group_by="protein_target",
).fit()

labels = clusterer.labels_frame()     # row-level; keep private by default
sizes = clusterer.cluster_sizes_frame()
fig, ax = clusterer.plot_correlation_heatmap("sfGFP")
```

## Methodological notes

- The default is Spearman correlation with average linkage. Always compare at least Pearson-volume and presence-based sensitivity analyses before treating clusters as robust.
- Features constant within a target group (e.g. fixed base buffer/lysate levels) are removed for that group's analysis. Entities with zero variance across remaining features are excluded and listed in the result.
- `1 - r` is used as a correlation dissimilarity. It is not guaranteed to be Euclidean, so Ward, centroid, and median linkage are intentionally disallowed.
- Reagent vectors are sparse and contain many zeros. Spearman can be influenced by tied ranks/common absences, and recipe volumes have a compositional/total-volume constraint. Check sensitivity and inspect cluster profiles rather than interpreting the dendrogram as a ground truth.
- Exact duplicate formulations may receive identical or nearly identical profiles; report duplicates and consider them when selecting experiments.
- Outputs from real competition data should remain outside the Git working tree by default. Obtain coordinator permission before sharing even aggregate figures or cluster summaries.

## Tests

From this folder:

```bash
python -m unittest discover -s tests -v
```

Tests use synthetic records and do not access the live database.


In [ ]:
%%writefile /content/htgaa-automation/tests/test_pairwise_correlation_clustering.py
import json
import unittest

import numpy as np

from analysis.cfps_correlation.pairwise_correlation_clustering import (
    CFPSSubmissionCorrelationAdapter,
    AbstractPairwiseCorrelationClusterer,
    adjusted_rand_index,
)


def _record(record_id, sample_id, target, reagents, batch="batch-a"):
    return {
        "id": record_id,
        "created": "2026-10-27 10:00:00.000Z",
        "design_json": json.dumps({
            "sample_id": sample_id,
            "sample_type": "experimental",
            "reagent_list": {"base_buffer": 2000, "lysate": 5000, **reagents},
            "metadata": {"protein_target": target, "batch_id": batch},
        }),
    }


class TestCFPSAdapter(unittest.TestCase):
    def setUp(self):
        self.records = [
            _record("id-a", "sample-a", "sfGFP", {"r1": 100, "r3": 50, "r5": 25}),
            _record("id-b", "sample-b", "sfGFP", {"r1": 200, "r3": 100, "r5": 50}),
            _record("id-c", "sample-c", "sfGFP", {"r2": 100, "r4": 50}),
            _record("id-d", "sample-d", "sfGFP", {"r2": 50, "r4": 100, "r5": 25}),
            _record("id-e", "sample-e", "PETase", {"r1": 10, "r2": 20, "r3": 5, "r4": 10}),
            _record("id-f", "sample-f", "PETase", {"r1": 20, "r2": 40, "r3": 10, "r4": 20}),
            _record("id-g", "sample-g", "PETase", {"r2": 10, "r3": 30, "r5": 50}),
        ]

    def test_abstract_contract(self):
        self.assertTrue(issubclass(CFPSSubmissionCorrelationAdapter, AbstractPairwiseCorrelationClusterer))

    def test_extracts_sparse_json_to_zero_filled_matrix(self):
        adapter = CFPSSubmissionCorrelationAdapter(self.records, group_by="protein_target")
        X = adapter.build_feature_matrix()
        self.assertEqual(X.loc["id-a", "r2"], 0.0)
        self.assertEqual(X.loc["id-a", "r1"], 100.0)
        self.assertIn("protein_target", adapter.build_metadata().columns)

    def test_constant_features_dropped_and_cluster_fit(self):
        adapter = CFPSSubmissionCorrelationAdapter(
            self.records, correlation_method="spearman", linkage_method="average", n_clusters=2,
            group_by="protein_target", min_features=3,
        ).fit()
        self.assertIn("sfGFP", adapter.results_)
        self.assertIn("PETase", adapter.results_)
        sf = adapter.results_["sfGFP"]
        self.assertIn("base_buffer", sf.dropped_features)
        self.assertIn("lysate", sf.dropped_features)
        self.assertGreater(sf.correlation_matrix.loc["id-a", "id-b"], 0.99)
        self.assertTrue(np.allclose(sf.distance_matrix.to_numpy(), sf.distance_matrix.to_numpy().T))
        self.assertEqual(len(adapter.labels_frame()), 7)

    def test_presence_mode(self):
        adapter = CFPSSubmissionCorrelationAdapter(self.records, feature_mode="presence", n_clusters=2).fit()
        self.assertEqual(adapter.results_["sfGFP"].feature_mode, "presence")
        self.assertTrue(set(np.unique(adapter.results_["sfGFP"].feature_matrix.to_numpy())).issubset({0.0, 1.0}))

    def test_bad_volume_fails_loudly(self):
        records = [_record("bad", "bad", "sfGFP", {"r1": "not-a-volume", "r2": 10, "r3": 25})]
        with self.assertRaisesRegex(ValueError, "invalid/nonnegative-volume"):
            CFPSSubmissionCorrelationAdapter(records)

    def test_snapshot_mapping_input(self):
        snapshot = {"collections": {"cfps_designs": self.records}}
        adapter = CFPSSubmissionCorrelationAdapter(snapshot)
        self.assertEqual(len(adapter.build_feature_matrix()), len(self.records))

    def test_agent_single_and_batch_payload_shapes(self):
        sample = {
            "sample_id": "agent-sample",
            "sample_type": "experimental",
            "reagent_list": {"r1": 100, "r2": 50, "r3": 25},
            "metadata": {"protein_target": "sfGFP"},
        }
        single = CFPSSubmissionCorrelationAdapter({"sample": sample})
        self.assertEqual(single.build_feature_matrix().loc["agent-sample", "r1"], 100.0)
        batch = CFPSSubmissionCorrelationAdapter({"batch_id": "batch-agent", "samples": [sample, {**sample, "sample_id": "agent-sample-2", "reagent_list": {"r1": 0, "r2": 100, "r3": 50}}]})
        self.assertEqual(len(batch.build_feature_matrix()), 2)
        self.assertTrue((batch.build_metadata()["batch_id"] == "batch-agent").all())

    def test_adjusted_rand_index_is_label_permutation_invariant(self):
        self.assertAlmostEqual(adjusted_rand_index([1, 1, 2, 2], ["x", "x", "y", "y"]), 1.0)
        self.assertLess(adjusted_rand_index([1, 1, 2, 2], ["x", "y", "x", "y"]), 0.0)

    def test_compare_with_reports_overlap_and_ari(self):
        a = CFPSSubmissionCorrelationAdapter(self.records, correlation_method="spearman", n_clusters=2).fit()
        b = CFPSSubmissionCorrelationAdapter(self.records, correlation_method="pearson", n_clusters=2).fit()
        comparison = a.compare_with(b)
        self.assertEqual(set(comparison["group"]), {"sfGFP", "PETase"})
        self.assertTrue((comparison["n_overlap"] > 0).all())


if __name__ == "__main__":
    unittest.main()


In [ ]:
# 12.2 Fit the primary clusterer on the fetched snapshot.
# Change these settings deliberately; do not compare cluster labels across runs without checking shared IDs.
import sys
import json
from datetime import datetime, timezone
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from analysis.cfps_correlation import CFPSSubmissionCorrelationAdapter

CORRELATION_METHOD = "spearman"   # "spearman" or "pearson"
FEATURE_MODE = "volume"           # "volume", "log1p_volume", or "presence"
LINKAGE_METHOD = "average"        # average/complete/single/weighted; Ward is not valid for 1-r dissimilarity
N_CLUSTERS_PER_TARGET = 6
MIN_VARIABLE_FEATURES = 3

clusterer = CFPSSubmissionCorrelationAdapter(
    snapshot,
    feature_mode=FEATURE_MODE,
    correlation_method=CORRELATION_METHOD,
    linkage_method=LINKAGE_METHOD,
    n_clusters=N_CLUSTERS_PER_TARGET,
    min_features=MIN_VARIABLE_FEATURES,
    group_by="protein_target",
).fit()

print("Clustered groups:", list(clusterer.results_))
print("Skipped groups:", clusterer.skipped_groups_ or "none")
for group_name, result in clusterer.results_.items():
    print(
        f"{group_name}: n={len(result.labels)}, features retained={result.feature_matrix.shape[1]}, "
        f"constant features dropped={len(result.dropped_features)}, "
        f"constant/invalid recipe vectors dropped={len(result.dropped_entities)}, "
        f"clusters={result.labels.nunique()}"
    )


In [ ]:
# 12.3 Save row-level matrices privately and render three diagnostics per target.
# These files contain row-level competition data/derived results. They are NOT written to REPORT_DIR.
CLUSTER_PRIVATE_DIR = PRIVATE_RUN_DIR / "pairwise_correlation_clustering"
cluster_files = clusterer.save_private_outputs(CLUSTER_PRIVATE_DIR)

cluster_figure_dir = CLUSTER_PRIVATE_DIR / "figures"
cluster_figure_dir.mkdir(parents=True, exist_ok=True)
for group_name, result in clusterer.results_.items():
    safe_name = clusterer._slug(group_name)
    fig, ax = clusterer.plot_correlation_heatmap(
        group_name,
        save_path=cluster_figure_dir / f"correlation_heatmap_{safe_name}_private.png",
    )
    plt.show()
    plt.close(fig)

    fig, ax = clusterer.plot_dendrogram(
        group_name,
        save_path=cluster_figure_dir / f"dendrogram_{safe_name}_private.png",
    )
    plt.show()
    plt.close(fig)

    fig, ax = clusterer.plot_cluster_sizes(
        group_name,
        save_path=cluster_figure_dir / f"cluster_sizes_{safe_name}_private.png",
    )
    plt.show()
    plt.close(fig)

clustering_manifest = {
    "created_utc": datetime.now(timezone.utc).isoformat(),
    "interpretation": "Pairwise correlations between submission recipe vectors across reagent features; not experimental-outcome correlations.",
    "source_collection": DESIGNS_COLLECTION,
    "source_repository_commit": REPO_SHA,
    "feature_mode": FEATURE_MODE,
    "correlation_method": CORRELATION_METHOD,
    "linkage_method": LINKAGE_METHOD,
    "requested_clusters_per_group": N_CLUSTERS_PER_TARGET,
    "min_variable_features": MIN_VARIABLE_FEATURES,
    "group_by": "protein_target",
    "groups": {
        name: {
            "n_entities_clustered": int(len(result.labels)),
            "n_features_retained": int(result.feature_matrix.shape[1]),
            "n_constant_features_dropped": int(len(result.dropped_features)),
            "n_constant_entity_vectors_dropped": int(len(result.dropped_entities)),
            "cluster_sizes": {str(k): int(v) for k, v in result.labels.value_counts().sort_index().items()},
        }
        for name, result in clusterer.results_.items()
    },
    "skipped_groups": clusterer.skipped_groups_,
    "private_row_level_outputs": True,
    "sharing_status": "Not approved for publication; inspect permissions and review outputs first.",
}
(CLUSTER_PRIVATE_DIR / "clustering_manifest_private.json").write_text(
    json.dumps(clustering_manifest, indent=2, ensure_ascii=False), encoding="utf-8"
)
print(f"Private clustering output directory: {CLUSTER_PRIVATE_DIR}")
print("Wrote", len(cluster_files), "private CSVs plus the manifest and figures.")


### Interpretation checklist before using the clusters

- Inspect the retained feature count and removed invariant reagents for each target. Constant components such as fixed buffer/lysate cannot separate submissions.
- The default Spearman method sees reagent patterns as within-recipe ranks; numerous zeros/tied ranks can inflate similarity. Presence-based clustering is a useful sensitivity check, not a definitive replacement.
- Recipe totals are constrained (currently 20,000 nL in the source contract), so ingredient amounts are not unconstrained independent variables. Correlation does not establish that one ingredient causes another's effect.
- Near-identical recipes and batch-related gradients can dominate clusters. Check duplicate formulations and batch metadata separately before selecting experiments.
- Inspect sensitivity across Spearman-volume, Pearson-volume, and Spearman-presence settings using adjusted Rand index (ARI) on overlapping submissions. ARI near 1 indicates similar partitions; low ARI means the cluster structure depends strongly on representation or metric.
- Do not interpret cluster membership as predicted yield or expected information gain. Those require outcome models/measurements and a separately validated selection objective.


In [ ]:
# 12.4 Sensitivity analysis: do clusters change under a different correlation/feature definition?
# This uses ARI on entities clustered by both runs; overlap fractions are reported by compare_with().
SENSITIVITY_CONFIGS = [
    ("pearson_volume", "pearson", "volume"),
    ("spearman_presence", "spearman", "presence"),
]
sensitivity_clusterers = {"primary": clusterer}
sensitivity_failures = {}
for label, method, feature_mode in SENSITIVITY_CONFIGS:
    candidate = CFPSSubmissionCorrelationAdapter(
        snapshot,
        feature_mode=feature_mode,
        correlation_method=method,
        linkage_method=LINKAGE_METHOD,
        n_clusters=N_CLUSTERS_PER_TARGET,
        min_features=MIN_VARIABLE_FEATURES,
        group_by="protein_target",
    )
    try:
        candidate.fit()
        sensitivity_clusterers[label] = candidate
    except Exception as exc:
        sensitivity_failures[label] = f"{type(exc).__name__}: {str(exc)[:250]}"

sensitivity_frames = []
for label, candidate in sensitivity_clusterers.items():
    if label == "primary":
        continue
    comparison = clusterer.compare_with(candidate)
    if not comparison.empty:
        comparison.insert(0, "comparison", f"primary_vs_{label}")
        sensitivity_frames.append(comparison)

if sensitivity_frames:
    sensitivity_df = pd.concat(sensitivity_frames, ignore_index=True)
    display(sensitivity_df)
    sensitivity_df.to_csv(CLUSTER_PRIVATE_DIR / "sensitivity_ari_private.csv", index=False)
else:
    sensitivity_df = pd.DataFrame()
    print("No successful sensitivity comparisons were available.")

(CLUSTER_PRIVATE_DIR / "sensitivity_failures_private.json").write_text(
    json.dumps(sensitivity_failures, indent=2), encoding="utf-8"
)
print("Sensitivity comparison files are private:", CLUSTER_PRIVATE_DIR)


## 13. Review the proposed Git changes before sharing

The notebook does **not** run `git add`, `git commit`, or `git push`. Review all outputs and confirm that only intended aggregate figures/tables are present.

```bash
cd /content/htgaa-automation
git status --short
git diff --stat
```

Recommended repository placement:

- Notebook: `notebooks/cfps_database_snapshot_and_exploration.ipynb`
- Reusable clustering module: `analysis/cfps_correlation/pairwise_correlation_clustering.py`
- Unit tests: `tests/test_pairwise_correlation_clustering.py`
- Aggregate outputs for review: `reports/cfps_exploration/`
- Raw snapshot / row-level tables: stay in `/content/cfps_private_snapshots/` and **never** be committed by default.

To save the notebook to the existing repository, use **Colab → File → Save a copy in GitHub**, choose `RonanChance/htgaa-automation`, use a new feature branch or the workflow Ronan approves, and select the path above. If GitHub/Colab does not let you write directly to the repository, download the notebook and open a pull request with it. Do not assume that repository write access includes permission to publish the underlying competition data.

## Verification / known limitations

- The direct `_superusers` authentication path, collection names, record ID, and submission JSON fields were identified in the repository's current source. Live authentication and record retrieval still depend on the credentials available in your Colab session.
- The catalogue parser is deliberately best-effort because the stored `cfps_reagent_groups` payload may be nested/stringified differently than expected. If it recognizes zero reagents, inspect the raw record privately and update the parser using that observed shape.
- The 20,000 nL total is a diagnostic from the repository's current CFPS design contract. Review historical records and contract changes before treating mismatches as invalid.
- The figures describe **submissions and recipe composition**, not experimental performance or expected information gain. They must not be interpreted as yield/fitness results unless real measurement data are explicitly linked and verified.
- No confidential-client secrets, Keycloak token, PB password, PB auth token, raw row-level submission export, author identifier, or free-text comment is written to the repo report directory.

The clustering module and tests are created in the checkout by the notebook cells. Review both before committing. The clustering matrices, row-level labels, and actual-data clustering figures stay in the private snapshot directory, not the repository reports directory. The cluster summaries/plots should not be published until Ronan confirms sharing permission.
